In [3]:
import cv2
import numpy as np

class GeneticAlgorithmSegmentation:
    def __init__(self, image_path, pop_size=20, generations=30, mutation_rate=0.05):
        self.img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if self.img is None:
            raise FileNotFoundError(f"Could not load image at {image_path}")

        self.hist, _ = np.histogram(self.img.ravel(), bins=256, range=(0, 256))
        self.prob = self.hist / self.img.size

        self.pop_size = pop_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.i_indices = np.arange(256)

    def fitness_otsu(self, threshold):
        T = int(threshold)

        w0 = np.sum(self.prob[:T])
        w1 = np.sum(self.prob[T:])

        if w0 == 0 or w1 == 0:
            return 0.0

        mu0 = np.sum(self.i_indices[:T] * self.prob[:T]) / w0
        mu1 = np.sum(self.i_indices[T:] * self.prob[T:]) / w1

        var_between = w0 * w1 * ((mu0 - mu1) ** 2)
        return float(var_between)

    def initialize_population(self):
        return np.random.randint(1, 255, size=self.pop_size)

    def select_parents(self, population, fitnesses):
        selected = []
        for _ in range(self.pop_size):
            i1, i2 = np.random.choice(len(population), size=2, replace=False)
            winner = population[i1] if fitnesses[i1] >= fitnesses[i2] else population[i2]
            selected.append(winner)
        return np.array(selected)

    def crossover(self, parent1, parent2):
        bin1 = format(parent1, '08b')
        bin2 = format(parent2, '08b')

        point = np.random.randint(1, 8)
        child1_bin = bin1[:point] + bin2[point:]
        child2_bin = bin2[:point] + bin1[point:]

        return int(child1_bin, 2), int(child2_bin, 2)

    def mutate(self, individual):
        bin_str = list(format(individual, '08b'))
        for i in range(len(bin_str)):
            if np.random.rand() < self.mutation_rate:
                bin_str[i] = '1' if bin_str[i] == '0' else '0'

        mutated_val = int("".join(bin_str), 2)
        return max(1, min(254, mutated_val))

    def run(self):
        population = self.initialize_population()
        best_threshold = population[0]
        best_fitness = -1

        for gen in range(self.generations):
            fitnesses = np.array([self.fitness_otsu(ind) for ind in population])

            max_idx = np.argmax(fitnesses)
            if fitnesses[max_idx] > best_fitness:
                best_fitness = fitnesses[max_idx]
                best_threshold = population[max_idx]

            parents = self.select_parents(population, fitnesses)

            next_pop = []
            for i in range(0, self.pop_size, 2):
                p1, p2 = parents[i], parents[(i + 1) % self.pop_size]
                c1, c2 = self.crossover(p1, p2)
                next_pop.extend([self.mutate(c1), self.mutate(c2)])

            population = np.array(next_pop[:self.pop_size])

        return best_threshold, best_fitness

    def segment_image(self, threshold):
        _, binary_mask = cv2.threshold(self.img, threshold, 255, cv2.THRESH_BINARY)
        foreground = cv2.bitwise_and(self.img, self.img, mask=binary_mask)
        background = cv2.bitwise_and(self.img, self.img, mask=cv2.bitwise_not(binary_mask))
        return binary_mask, foreground, background


if __name__ == "__main__":
    ga_segmenter = GeneticAlgorithmSegmentation('/content/filip-kvasnak-voG9sYih3ds-unsplash.jpg', pop_size=25, generations=40, mutation_rate=0.08)
    optimal_T, best_var = ga_segmenter.run()

    mask, fg, bg = ga_segmenter.segment_image(optimal_T)
    cv2.imwrite('segmented_mask.png', mask)
    cv2.imwrite('foreground.png', fg)
    cv2.imwrite('background.png', bg)